In [35]:
import pandas as pd
import numpy as np

from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("synthetic_fraud_dataset.csv")

print("Dataset Loaded Successfully!")
print("Dataset Shape:", df.shape)

target = "Fraud_Label"

X = df.drop(columns=[target])
y = df[target]

print("\nFeatures Shape:", X.shape)
print("Target Shape:", y.shape)

X = X.drop(
    columns=["Transaction_ID", "User_ID"],
    errors="ignore"
)

X["Timestamp"] = pd.to_datetime(X["Timestamp"])

X["Year"] = X["Timestamp"].dt.year
X["Month"] = X["Timestamp"].dt.month
X["Day"] = X["Timestamp"].dt.day
X["Hour"] = X["Timestamp"].dt.hour
X["DayOfWeek"] = X["Timestamp"].dt.dayofweek

X = X.drop(columns=["Timestamp"])

categorical_columns = X.select_dtypes(
    include=["object", "str"]
).columns

print("\nCategorical Columns:")
print(categorical_columns.tolist())

for column in categorical_columns:
    encoder = LabelEncoder()
    X[column] = encoder.fit_transform(
        X[column].astype(str)
    )

print("\nTarget Distribution:")
print(y.value_counts())

model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    eval_metric="logloss"
)

print("\nXGBoost Model Created Successfully!")

scores = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print("\n" + "=" * 55)
print("5-FOLD CROSS-VALIDATION RESULTS")
print("=" * 55)

for i, score in enumerate(scores, start=1):
    print(
        f"Fold {i} Accuracy: {score:.4f} "
        f"({score * 100:.2f}%)"
    )

average_accuracy = scores.mean()
standard_deviation = scores.std()

print("-" * 55)
print(f"Average Accuracy: {average_accuracy:.4f}")
print(f"Average Accuracy: {average_accuracy * 100:.2f}%")
print(f"Standard Deviation: {standard_deviation:.4f}")
print("=" * 55)

Dataset Loaded Successfully!
Dataset Shape: (50000, 21)

Features Shape: (50000, 20)
Target Shape: (50000,)

Categorical Columns:
['Transaction_Type', 'Device_Type', 'Location', 'Merchant_Category', 'Card_Type', 'Authentication_Method']

Target Distribution:
Fraud_Label
0    33933
1    16067
Name: count, dtype: int64

XGBoost Model Created Successfully!

5-FOLD CROSS-VALIDATION RESULTS
Fold 1 Accuracy: 0.9984 (99.84%)
Fold 2 Accuracy: 0.9996 (99.96%)
Fold 3 Accuracy: 1.0000 (100.00%)
Fold 4 Accuracy: 0.9995 (99.95%)
Fold 5 Accuracy: 0.9991 (99.91%)
-------------------------------------------------------
Average Accuracy: 0.9993
Average Accuracy: 99.93%
Standard Deviation: 0.0005
